# 🩺 Diabetic Retinopathy Detection from Fundus Images
### ResNet-50 vs EfficientNet-B4 | Grad-CAM | Folder-based Dataset

---
**Goal**: Classify retinal fundus images into 5 severity grades of DR  
**Labels**: 0 = No DR, 1 = Mild, 2 = Moderate, 3 = Severe, 4 = Proliferative DR  
**Metric**: Quadratic Weighted Kappa (QWK)  
**Dataset**: sovitrath/diabetic-retinopathy-224x224-2019-data (colored_images folder)

## ⚙️ Stage 1: Setup & Imports

In [1]:
!pip install -q timm albumentations scikit-learn

import os, gc, cv2, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    cohen_kappa_score, classification_report,
    confusion_matrix, accuracy_score
)

warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device      : {DEVICE}')
print(f'✅ PyTorch     : {torch.__version__}')
if torch.cuda.is_available():
    print(f'✅ GPU         : {torch.cuda.get_device_name(0)}')
    print(f'✅ GPU Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

  error: subprocess-exited-with-error
  
  × Building wheel for stringzilla (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [10 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-314\cli
      copying cli\split.py -> build\lib.win-amd64-cpython-314\cli
      copying cli\wc.py -> build\lib.win-amd64-cpython-314\cli
      copying cli\__init__.py -> build\lib.win-amd64-cpython-314\cli
      running build_ext
      building 'stringzilla' extension
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for stringzilla
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> stringzilla


ModuleNotFoundError: No module named 'matplotlib'

## 📦 Stage 2: Data Collection & Understanding

In [2]:
# ── Dataset paths (colored_images folder structure) ────────────────────────────
BASE_DIR = Path('/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-2019-data/colored_images')

# Folder name → integer label
FOLDER_TO_LABEL = {
    'No_DR'         : 0,
    'Mild'          : 1,
    'Moderate'      : 2,
    'Severe'        : 3,
    'Proliferate_DR': 4,
}

CLASS_NAMES  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
CLASS_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

# ── Hyperparameters ────────────────────────────────────────────────────────────
CFG = {
    'img_size'    : 224,
    'batch_size'  : 32,
    'num_classes' : 5,
    'epochs'      : 20,
    'lr'          : 1e-4,
    'weight_decay': 1e-4,
    'num_workers' : 2,
    'n_folds'     : 5,
    'use_fold'    : 0,
    'label_col'   : 'diagnosis',
}

# ── Build DataFrame from folder structure ──────────────────────────────────────
records = []
for folder_name, label in FOLDER_TO_LABEL.items():
    folder_path = BASE_DIR / folder_name
    if not folder_path.exists():
        print(f'⚠️  Folder not found: {folder_path}')
        continue
    for img_path in folder_path.glob('*'):
        if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
            records.append({
                'filepath' : str(img_path),
                'diagnosis': label,
                'class'    : CLASS_NAMES[label]
            })

df = pd.DataFrame(records)
print(f'📊 Total images found : {len(df)}')
print(f'\n📈 Label distribution :')
for label, name in enumerate(CLASS_NAMES):
    count = (df['diagnosis'] == label).sum()
    print(f'  Class {label} ({name:20s}): {count:5d} images  ({count/len(df)*100:.1f}%)')

print(f'\n🔍 Sample rows:')
display(df.head())

NameError: name 'Path' is not defined

## 🧹 Stage 3: Data Preprocessing & Cleaning

In [ ]:
# ── Validate files exist ───────────────────────────────────────────────────────
missing = df['filepath'].apply(lambda p: not Path(p).exists()).sum()
print(f'❌ Missing images : {missing}')
print(f'✅ Valid  images  : {len(df) - missing}')

df = df[df['filepath'].apply(lambda p: Path(p).exists())].reset_index(drop=True)
print(f'📊 Clean dataset  : {len(df)} images')

# ── Stratified K-Fold split ───────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=SEED)
df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(df, df['diagnosis'])):
    df.loc[val_idx, 'fold'] = fold

train_df = df[df['fold'] != CFG['use_fold']].reset_index(drop=True)
val_df   = df[df['fold'] == CFG['use_fold']].reset_index(drop=True)

print(f'\n✅ Train size : {len(train_df)}')
print(f'✅ Val   size : {len(val_df)}')
print(f'\nTrain label counts:')
print(train_df['diagnosis'].value_counts().sort_index())

## 🔍 Stage 4: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('📊 DR Dataset — EDA', fontsize=16, fontweight='bold')

counts = df['diagnosis'].value_counts().sort_index()

# Bar chart
axes[0].bar(CLASS_NAMES, counts.values, color=CLASS_COLORS, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('DR Grade')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=9, fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=CLASS_NAMES, colors=CLASS_COLORS,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 9})
axes[1].set_title('Class Proportions')

# Imbalance ratio
imbalance = counts.max() / counts
axes[2].barh(CLASS_NAMES, imbalance.values, color=CLASS_COLORS, edgecolor='black')
axes[2].set_title('Imbalance Ratio (vs Majority Class)')
axes[2].set_xlabel('Ratio')
for i, v in enumerate(imbalance.values):
    axes[2].text(v + 0.05, i, f'{v:.1f}x', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: eda_class_distribution.png')

In [ ]:
# ── Sample images per class ────────────────────────────────────────────────────
fig, axes = plt.subplots(5, 5, figsize=(18, 18))
fig.suptitle('Sample Fundus Images per DR Grade', fontsize=16, fontweight='bold')

for cls in range(5):
    samples = df[df['diagnosis'] == cls].sample(5, random_state=SEED)
    for j, (_, row) in enumerate(samples.iterrows()):
        img = cv2.imread(row['filepath'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[cls][j].imshow(img)
        axes[cls][j].axis('off')
        if j == 0:
            axes[cls][j].set_ylabel(
                f'Grade {cls}\n{CLASS_NAMES[cls]}',
                fontsize=10, rotation=0, labelpad=80, va='center'
            )

plt.tight_layout()
plt.savefig('eda_sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print('💾 Saved: eda_sample_images.png')

In [ ]:
# ── Image size check ──────────────────────────────────────────────────────────
sample_imgs = df.sample(min(200, len(df)), random_state=SEED)
heights, widths = [], []
for _, row in tqdm(sample_imgs.iterrows(), total=len(sample_imgs), desc='Checking sizes'):
    img = cv2.imread(row['filepath'])
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h); widths.append(w)

print(f'📐 Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}')
print(f'📐 Width : min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}')
print(f'→ All images will be resized to {CFG["img_size"]}x{CFG["img_size"]}')

## 🔧 Stage 5: Feature Engineering & Data Augmentation

In [ ]:
# ── Ben Graham preprocessing ──────────────────────────────────────────────────
def ben_graham_preprocess(img_bgr, sigmaX=10):
    """Enhance fine retinal features by subtracting local average."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    enhanced = cv2.addWeighted(
        img_rgb, 4,
        cv2.GaussianBlur(img_rgb, (0, 0), sigmaX), -4,
        128
    )
    return enhanced


# ── Dataset class ─────────────────────────────────────────────────────────────
class DRDataset(Dataset):
    def __init__(self, df, transform=None, use_ben_graham=True):
        self.df             = df.reset_index(drop=True)
        self.transform      = transform
        self.use_ben_graham = use_ben_graham

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_bgr = cv2.imread(row['filepath'])

        if self.use_ben_graham:
            img = ben_graham_preprocess(img_bgr)
        else:
            img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)['image']

        label = int(row['diagnosis'])
        return img, torch.tensor(label, dtype=torch.long)


# ── Augmentation pipelines ────────────────────────────────────────────────────
IMG_SIZE = CFG['img_size']

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, fill_value=0, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

print('✅ Dataset class and augmentation pipeline ready.')

# ── Visualise augmentations ───────────────────────────────────────────────────
sample_row = train_df.iloc[0]
orig_bgr   = cv2.imread(sample_row['filepath'])
orig_rgb   = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
ben_img    = ben_graham_preprocess(orig_bgr)

individual_augs = [
    ('H-Flip',        A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1)])),
    ('Brightness',    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.RandomBrightnessContrast(p=1)])),
    ('ShiftScaleRot', A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.ShiftScaleRotate(p=1)])),
    ('Dropout',       A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.CoarseDropout(max_holes=8, max_height=24, max_width=24, p=1)])),
]

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
fig.suptitle('Augmentation Examples', fontsize=14, fontweight='bold')
axes[0].imshow(cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))); axes[0].set_title('Original');    axes[0].axis('off')
axes[1].imshow(cv2.resize(ben_img,  (IMG_SIZE, IMG_SIZE))); axes[1].set_title('Ben Graham'); axes[1].axis('off')
for i, (title, aug) in enumerate(individual_augs):
    aug_img = aug(image=orig_rgb)['image']
    axes[i+2].imshow(aug_img); axes[i+2].set_title(title); axes[i+2].axis('off')

plt.tight_layout()
plt.savefig('augmentations.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Saved: augmentations.png')

In [ ]:
# ── Class weights & DataLoaders ───────────────────────────────────────────────
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(CFG['num_classes']),
    y=train_df['diagnosis'].values
)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

print('⚖️  Class weights (to handle imbalance):')
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f'  Class {i} ({name:20s}): {w:.4f}')


def get_loaders(train_df, val_df):
    train_ds = DRDataset(train_df, transform=train_transform)
    val_ds   = DRDataset(val_df,   transform=val_transform)

    # WeightedRandomSampler — oversample rare classes
    sample_weights = class_weights[train_df['diagnosis'].values]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_ds, batch_size=CFG['batch_size'],
        sampler=sampler, num_workers=CFG['num_workers'], pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG['batch_size'],
        shuffle=False, num_workers=CFG['num_workers'], pin_memory=True
    )
    return train_loader, val_loader


train_loader, val_loader = get_loaders(train_df, val_df)
imgs, labels = next(iter(train_loader))
print(f'\n✅ Train batches : {len(train_loader)}')
print(f'✅ Val   batches : {len(val_loader)}')
print(f'✅ Batch shape   : {imgs.shape}  labels: {labels.shape}')

## 🏗️ Stage 6: Model Building & Training

In [ ]:
# ── Model factory ─────────────────────────────────────────────────────────────
def build_model(model_name, num_classes=5, pretrained=True):
    if model_name == 'resnet50':
        model = timm.create_model('resnet50', pretrained=pretrained)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    elif model_name == 'efficientnet_b4':
        model = timm.create_model('efficientnet_b4', pretrained=pretrained)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    else:
        raise ValueError(f'Unknown model: {model_name}')
    return model.to(DEVICE)


# ── Training loop helpers ─────────────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, scheduler):
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []

    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    scheduler.step()
    n = len(loader.dataset)
    return (
        running_loss / n,
        accuracy_score(all_labels, all_preds),
        cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    )


@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    running_loss, all_preds, all_labels = 0.0, [], []

    for imgs, labels in tqdm(loader, desc='  Val  ', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    n = len(loader.dataset)
    return (
        running_loss / n,
        accuracy_score(all_labels, all_preds),
        cohen_kappa_score(all_labels, all_preds, weights='quadratic'),
        all_preds,
        all_labels
    )


def train_model(model_name, train_loader, val_loader, epochs=CFG['epochs']):
    print(f'\n{"="*65}')
    print(f'  🚀  Training: {model_name.upper()}')
    print(f'{"="*65}')

    model     = build_model(model_name)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'],
                             weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    history = {k: [] for k in
               ['train_loss','val_loss','train_acc','val_acc','train_kappa','val_kappa']}

    best_kappa  = -1.0
    best_preds  = None
    best_labels = None
    save_path   = f'best_{model_name}.pth'

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, tr_kappa = train_epoch(
            model, train_loader, criterion, optimizer, scheduler)
        vl_loss, vl_acc, vl_kappa, vl_preds, vl_labels = val_epoch(
            model, val_loader, criterion)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        history['train_kappa'].append(tr_kappa)
        history['val_kappa'].append(vl_kappa)

        marker = ''
        if vl_kappa > best_kappa:
            best_kappa  = vl_kappa
            best_preds  = vl_preds
            best_labels = vl_labels
            torch.save(model.state_dict(), save_path)
            marker = '  ✅ BEST'

        print(f'Epoch {epoch:02d}/{epochs}  '
              f'| Train  loss={tr_loss:.4f}  acc={tr_acc:.4f}  QWK={tr_kappa:.4f}'
              f'  | Val  loss={vl_loss:.4f}  acc={vl_acc:.4f}  QWK={vl_kappa:.4f}'
              f'{marker}')

    print(f'\n🏆 Best Val QWK [{model_name}]: {best_kappa:.4f}')
    print(f'💾 Saved: {save_path}')
    return model, history, best_kappa, best_preds, best_labels, save_path

print('✅ Model factory and training loop ready.')

In [ ]:
# ── Train ResNet-50 ───────────────────────────────────────────────────────────
resnet_model, resnet_history, resnet_best_kappa, \
resnet_preds, resnet_labels, resnet_path = train_model(
    'resnet50', train_loader, val_loader
)

In [ ]:
# ── Train EfficientNet-B4 ─────────────────────────────────────────────────────
effnet_model, effnet_history, effnet_best_kappa, \
effnet_preds, effnet_labels, effnet_path = train_model(
    'efficientnet_b4', train_loader, val_loader
)

## 📊 Stage 7: Model Evaluation & Comparison

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
fig.suptitle('Training Curves: ResNet-50 vs EfficientNet-B4', fontsize=15, fontweight='bold')

plot_specs = [
    ('val_loss',   'train_loss',   'Validation Loss'),
    ('val_acc',    'train_acc',    'Validation Accuracy'),
    ('val_kappa',  'train_kappa',  'Quadratic Weighted Kappa (Val)'),
]

e = range(1, len(resnet_history['val_loss']) + 1)
for ax, (val_key, tr_key, title) in zip(axes, plot_specs):
    ax.plot(e, resnet_history[val_key], 'b-o', markersize=4, label='ResNet-50 val')
    ax.plot(e, effnet_history[val_key],  'r-s', markersize=4, label='EfficientNet-B4 val')
    ax.plot(e, resnet_history[tr_key],  'b--',  alpha=0.4,   label='ResNet-50 train')
    ax.plot(e, effnet_history[tr_key],   'r--',  alpha=0.4,   label='EfficientNet-B4 train')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: training_curves.png')

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
def plot_cm(preds, labels, model_name, ax):
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, cbar=False)
    qwk = cohen_kappa_score(labels, preds, weights='quadratic')
    ax.set_title(f'{model_name}  |  QWK = {qwk:.4f}', fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=30)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
plot_cm(resnet_preds, resnet_labels, 'ResNet-50',       axes[0])
plot_cm(effnet_preds, effnet_labels, 'EfficientNet-B4', axes[1])
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: confusion_matrices.png')

In [ ]:
# ── Classification reports ────────────────────────────────────────────────────
print('='*65)
print('📋 ResNet-50 — Classification Report')
print('='*65)
print(classification_report(resnet_labels, resnet_preds, target_names=CLASS_NAMES))

print('='*65)
print('📋 EfficientNet-B4 — Classification Report')
print('='*65)
print(classification_report(effnet_labels, effnet_preds, target_names=CLASS_NAMES))

# ── Final comparison table ────────────────────────────────────────────────────
resnet_acc = accuracy_score(resnet_labels, resnet_preds)
effnet_acc = accuracy_score(effnet_labels, effnet_preds)

comparison = pd.DataFrame({
    'Model'          : ['ResNet-50', 'EfficientNet-B4'],
    'Best Val QWK'   : [round(resnet_best_kappa, 4), round(effnet_best_kappa, 4)],
    'Val Accuracy'   : [round(resnet_acc, 4),        round(effnet_acc, 4)],
    'Parameters (M)' : [25.6, 19.3],
    'Winner'         : [
        '🏆' if resnet_best_kappa > effnet_best_kappa else '',
        '🏆' if effnet_best_kappa >= resnet_best_kappa else ''
    ]
})
print('\n📊 Model Comparison:')
display(comparison)

# ── Pick best model ───────────────────────────────────────────────────────────
if effnet_best_kappa >= resnet_best_kappa:
    BEST_MODEL_NAME = 'efficientnet_b4'
    BEST_MODEL      = effnet_model
    BEST_PATH       = effnet_path
    BEST_PREDS      = effnet_preds
    BEST_LABELS     = effnet_labels
else:
    BEST_MODEL_NAME = 'resnet50'
    BEST_MODEL      = resnet_model
    BEST_PATH       = resnet_path
    BEST_PREDS      = resnet_preds
    BEST_LABELS     = resnet_labels

print(f'\n🏆 WINNER: {BEST_MODEL_NAME}  (QWK = {max(resnet_best_kappa, effnet_best_kappa):.4f})')

## 🔬 Stage 8: Model Interpretation with Grad-CAM

In [ ]:
# ── Pure-PyTorch Grad-CAM (no external library) ───────────────────────────────
class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.gradients    = None
        self.activations  = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach())
        )
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach())
        )

    def generate(self, img_tensor):
        self.model.eval()
        x = img_tensor.unsqueeze(0).to(DEVICE)
        out = self.model(x)
        pred_cls = out.argmax(dim=1).item()
        probs    = out.softmax(dim=1)[0].detach().cpu().numpy()

        self.model.zero_grad()
        out[0, pred_cls].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1)).squeeze().cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        cam = cam - cam.min()
        if cam.max() > 0:
            cam /= cam.max()
        return cam, pred_cls, probs


def overlay_cam(img_rgb, cam, alpha=0.45):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return np.clip((1 - alpha) * img_rgb + alpha * heatmap, 0, 255).astype(np.uint8)


# ── Load best model weights & set target layer ────────────────────────────────
BEST_MODEL.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
BEST_MODEL.eval()

if BEST_MODEL_NAME == 'resnet50':
    target_layer = BEST_MODEL.layer4[-1]
else:
    target_layer = BEST_MODEL.blocks[-1][-1]

grad_cam = GradCAM(BEST_MODEL, target_layer)
print(f'✅ Grad-CAM ready on {BEST_MODEL_NAME}')

In [ ]:
# ── Visualise one sample per class ────────────────────────────────────────────
fig, axes = plt.subplots(5, 3, figsize=(12, 20))
fig.suptitle(f'Grad-CAM — {BEST_MODEL_NAME}', fontsize=14, fontweight='bold')

for ax, t in zip(axes[0], ['Original', 'Grad-CAM Heatmap', 'Overlay']):
    ax.set_title(t, fontsize=11, fontweight='bold')

for cls in range(5):
    sample  = val_df[val_df['diagnosis'] == cls].sample(1, random_state=SEED).iloc[0]
    orig_bgr = cv2.imread(sample['filepath'])
    orig_rgb  = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    ben_img   = ben_graham_preprocess(orig_bgr)

    t_img = val_transform(image=ben_img)['image']
    cam, pred_cls, probs = grad_cam.generate(t_img)

    disp   = cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))
    hmap   = (mpl_cm.jet(cam)[..., :3] * 255).astype(np.uint8)
    over   = overlay_cam(disp, cam)

    for ax, img in zip(axes[cls], [disp, hmap, over]):
        ax.imshow(img); ax.axis('off')

    axes[cls][0].set_ylabel(
        f'True: {CLASS_NAMES[cls]}\nPred: {CLASS_NAMES[pred_cls]}\n({probs[pred_cls]*100:.1f}%)',
        fontsize=8, rotation=0, labelpad=110, va='center'
    )

plt.tight_layout()
plt.savefig('gradcam_visualisations.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Saved: gradcam_visualisations.png')

## 🚀 Stage 9: Streamlit Deployment App

In [ ]:
app_code = '''
# ── app.py  ——  Streamlit DR Detection App ────────────────────────────────────
# Run locally: streamlit run app.py

import streamlit as st
import torch, torch.nn.functional as F
import timm, cv2, numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

CLASS_NAMES  = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
CLASS_COLORS = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
IMG_SIZE     = 224
DEVICE       = torch.device("cpu")

@st.cache_resource
def load_model():
    import torch.nn as nn
    model = timm.create_model("efficientnet_b4", pretrained=False)
    in_f  = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(in_f, 512),
        nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 5)
    )
    model.load_state_dict(torch.load("best_efficientnet_b4.pth", map_location=DEVICE))
    return model.eval()

def preprocess(img_np):
    img = cv2.addWeighted(img_np, 4, cv2.GaussianBlur(img_np,(0,0),10), -4, 128)
    t   = A.Compose([A.Resize(IMG_SIZE,IMG_SIZE),
                     A.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
                     ToTensorV2()])
    return t(image=img)["image"].unsqueeze(0)

st.set_page_config(page_title="DR Detection", page_icon="👁", layout="centered")
st.title("👁 Diabetic Retinopathy Detection")
st.caption("Upload a retinal fundus image — model will classify the DR severity grade.")

model    = load_model()
uploaded = st.file_uploader("Upload fundus image", type=["jpg","jpeg","png"])

if uploaded:
    img = np.array(Image.open(uploaded).convert("RGB"))
    st.image(img, caption="Uploaded Image", use_column_width=True)

    with st.spinner("Analysing image..."):
        with torch.no_grad():
            probs = F.softmax(model(preprocess(img)), dim=1)[0].numpy()

    pred = int(probs.argmax())
    st.markdown(f"### Result: **{CLASS_NAMES[pred]}** (Grade {pred})")
    st.markdown(f"**Confidence:** {probs[pred]*100:.1f}%")
    st.divider()
    st.markdown("#### Probability per class")
    for name, p, color in zip(CLASS_NAMES, probs, CLASS_COLORS):
        st.progress(float(p), text=f"{name}: {p*100:.1f}%")
'''

with open('app.py', 'w') as f:
    f.write(app_code)
print('💾 Streamlit app saved as: app.py')

## 📝 Stage 10: Final Summary

In [ ]:
best_qwk = max(resnet_best_kappa, effnet_best_kappa)

print('='*65)
print('  🎯  FINAL PROJECT SUMMARY — Diabetic Retinopathy Detection')
print('='*65)
print(f"""
📦  Dataset
    Source   : sovitrath/diabetic-retinopathy-224x224-2019-data
    Structure: colored_images/{{No_DR,Mild,Moderate,Severe,Proliferate_DR}}
    Classes  : 5  (No DR → Proliferative DR)
    Total    : {len(df)} images
    Split    : {CFG['n_folds']}-Fold Stratified CV  (fold {CFG['use_fold']} used)

🧹  Preprocessing
    • Ben Graham enhancement (local contrast boost)
    • Resize → {CFG['img_size']}x{CFG['img_size']}
    • ImageNet normalisation

📈  Augmentation  (train only)
    • Horizontal/Vertical flip, Random rotate 90°
    • ShiftScaleRotate (±30°), Brightness/Contrast jitter
    • HueSaturationValue, CoarseDropout (cutout)
    • WeightedRandomSampler for class imbalance

⚖️   Imbalance Handling
    • Weighted CrossEntropy loss  (sklearn balanced weights)
    • WeightedRandomSampler in DataLoader

🏗️   Models  (both pretrained ImageNet → fine-tuned)
    • ResNet-50      : custom head  Dropout→Linear512→ReLU→Dropout→Linear5
    • EfficientNet-B4: same head
    • Optimiser      : AdamW  (lr={CFG['lr']}, wd={CFG['weight_decay']})
    • Scheduler      : CosineAnnealingLR  (T_max={CFG['epochs']})
    • Grad clipping  : max_norm=1.0

📊  Results
    ResNet-50        Val QWK = {resnet_best_kappa:.4f}   Acc = {accuracy_score(resnet_labels, resnet_preds):.4f}
    EfficientNet-B4  Val QWK = {effnet_best_kappa:.4f}   Acc = {accuracy_score(effnet_labels, effnet_preds):.4f}

🏆  WINNER : {BEST_MODEL_NAME}   (QWK = {best_qwk:.4f})

🔬  Interpretability
    • Custom Grad-CAM (pure PyTorch, no external lib)
    • Heatmaps highlight lesion regions per class

🚀  Deployment
    • Streamlit app (app.py) — upload image → get prediction

📁  Output files
    eda_class_distribution.png
    eda_sample_images.png
    augmentations.png
    training_curves.png
    confusion_matrices.png
    gradcam_visualisations.png
    best_resnet50.pth
    best_efficientnet_b4.pth
    app.py
""")